In [68]:
import matplotlib.pyplot as plt
import polars as pl
import pandas as pd
import numpy as np
import json
import os
import sys

sys.path.append("..")

import seaborn as sns

from settings import (
    random_state,
    PROJECT_PATH,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
)
from catboost import CatBoostClassifier, Pool
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import recall_score, precision_score, f1_score
import mlflow
from mlflow import MlflowClient

This solution is based on the following version of MLflow

In [2]:
mlflow.__version__

'2.15.1'

In [3]:
def perform_cross_validation(
    X: pl.DataFrame,
    y: pl.Series,
    model,
    cross_val_type,
    scoring_metrics: tuple,
    groups=None,
):
    scores = cross_validate(
        model,
        X.to_numpy(),
        y.to_numpy(),
        cv=cross_val_type,
        return_train_score=True,
        return_estimator=True,
        scoring=scoring_metrics,
        groups=groups,
    )

    scores_dict = {}
    for metric in scoring_metrics:
        scores_dict["average_train_" + metric] = np.mean(scores["train_" + metric])
        scores_dict["train_" + metric + "_std"] = np.std(scores["train_" + metric])
        scores_dict["average_test_" + metric] = np.mean(scores["test_" + metric])
        scores_dict["test_" + metric + "_std"] = np.std(scores["test_" + metric])

    model.fit(X.to_numpy(), y.to_numpy())

    return scores, scores_dict, model

def get_features_most_importance(importances, feature_names, threshold=0.8):
    sorted_indices = np.argsort(importances)
    sorted_importances = importances[sorted_indices][::-1]
    sorted_feature_names = [feature_names[i] for i in sorted_indices][::-1]

    cumulated_importance = 0
    important_features = []

    for importance, feature_name in zip(sorted_importances, sorted_feature_names):
        cumulated_importance += importance
        important_features.append(feature_name)

        if cumulated_importance >= threshold:
            break

    return important_features

In [4]:
transactions = pl.read_parquet(
    os.path.join(PROJECT_PATH, "transactions_post_feature_engineering.parquet")
)


with open("../features_used.json", "r") as f:
    feature_names = json.load(f)

with open("../categorical_features_used.json", "r") as f:
    categorical_features = json.load(f)

numerical_features = [col for col in feature_names if col not in categorical_features]

In [90]:
transactions_v1 = transactions.filter(pl.col("annee_transaction") < 2020)

transactions_v2 = transactions.filter(
    pl.col("annee_transaction").is_between(2020, 2021)
)
features_1 = [
    "type_batiment_Appartement",
    "surface_habitable",
    "prix_m2_moyen_mois_precedent",
    "nb_transactions_mois_precedent",
    "taux_interet",
    "variation_taux_interet",
    "acceleration_taux_interet",
]

features_2 = features_1 + ["longitude", "latitude", "vefa"]

<b>Important:</b> Make sure to run this code in the same folder where the pre-Covid models were created. When you launch the MLflow UI (and a MlflowClient object) without additional arguments, the package uses the current folder as the model registry. Specifically, MLflow creates an mlruns folder where all your models are stored. 

Il suffit alors de : 
* lancer dans votre Terminal la commande mlflow ui en précisant l'adresse du dossier mlruns comme ceci : mlflow ui --backend-store-uri adresse_de_votre_choix
* Réaliser la même opération côté Python avec la commande mlflow.set_tracking_uri("adresse_de_votre_choix")

We declare new tag values to stay consistent with the context: new post-Covid feature engineering

In [6]:
experiment_tags = {
    "region": "Nouvelle-Aquitaine",
    "revision_de_donnees": "v2",
    "date_de_construction": "Fin 2021",
}

We tell MLflow that we will work with the Nouvelle-Aquitaine experiment as shown in the course.

In [53]:

client = MlflowClient(tracking_uri="http://127.0.0.1:5000")
nouvelle_acquitaine_experiment = mlflow.set_experiment("Nouvelle-Aquitaine")



We can retrieve the experiment id by printing the Experiment object

In [19]:
nouvelle_acquitaine_experiment_id = '247887458207936819'

In [10]:
transactions_post_covid_nouvelle_acquitaine = transactions_v2.filter(
    pl.col("nom_region_Nouvelle-Aquitaine") == 1
)

To compare like with like, we first need to load the models trained on the pre-Covid period, run inference on the new data, and observe the results. It would make little sense to compare a new model trained on new data with an old model trained on old data. 

To do this, we use the search_runs feature. We will use the experiment id to find the URI (storage address) of the best-performing model from the pre-Covid period. 

In [59]:
# This is a DataFrame 
all_experiments = mlflow.search_runs(search_all_experiments=True)

In [60]:
all_experiments.columns

Index(['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time',
       'end_time', 'metrics.train_precision_std',
       'metrics.average_train_recall', 'metrics.average_test_precision',
       'metrics.average_test_recall', 'metrics.train_recall_std',
       'metrics.average_train_precision', 'metrics.test_precision_std',
       'metrics.test_f1_std', 'metrics.average_train_f1',
       'metrics.average_test_f1', 'metrics.train_f1_std',
       'metrics.test_recall_std', 'params.features', 'params.random_state',
       'tags.mlflow.user', 'tags.mlflow.source.name',
       'tags.mlflow.log-model.history', 'tags.mlflow.runName',
       'tags.mlflow.source.type'],
      dtype='object')

For illustration purposes, we use a simple approach and select the model with the best average F1 score. A more robust approach would involve comparing train and test scores to check for overfitting and examining the standard deviation of the scores. 

In [24]:
highest_f1_score = all_experiments["metrics.average_test_f1"].max()

In [25]:
best_precovid_model = all_experiments.loc[
    (all_experiments["experiment_id"] == nouvelle_acquitaine_experiment_id) 
    & (all_experiments["metrics.average_test_f1"] == highest_f1_score)
    
]

We then retrieve the Run ID of the model, from which we can derive its storage address. 

In [44]:
best_precovid_model_run_id = best_precovid_model["run_id"].values[0] # In case multiple models share the same performance

This command identifies the model file name. It is the same name that was stored by the log_model method

In [62]:
client.list_artifacts(best_precovid_model_run_id)

[<FileInfo: file_size=None, is_dir=True, path='catboost_classifier'>]

In [64]:
model_local_path = mlflow.artifacts.download_artifacts(
  run_id=best_precovid_model_run_id, artifact_path="catboost_classifier"
)

In [66]:
best_precovid_model = mlflow.sklearn.load_model(model_local_path)

We then define a Run that will run inference of the pre-Covid model on the post-Covid data.

In [72]:

with mlflow.start_run(run_name="pre_covid_catboost_Nouvelle-Aquitaine_post_covid_data") as run:

    X = transactions_post_covid_nouvelle_acquitaine.drop(
        [REGRESSION_TARGET, CLASSIFICATION_TARGET]
    ).to_pandas()
    y_classification = transactions_post_covid_nouvelle_acquitaine[CLASSIFICATION_TARGET].to_pandas()

    classification_scoring_metrics = ["recall", "precision", "f1"]

    predictions = best_precovid_model.predict(Pool(X[features_1],y_classification))

    scores_dict = {}
    scores_dict["recall"] = recall_score(y_classification, predictions)
    scores_dict["precision"] = precision_score(y_classification, predictions)
    scores_dict["f1"] = f1_score(y_classification, predictions)

    mlflow.log_param("random_state", random_state)
    mlflow.log_param("features", features_1)

    for metric, value in scores_dict.items():
        mlflow.log_metric(metric, value)

    mlflow.sklearn.log_model(best_precovid_model, "best_pre_covid_model_post_covid_data")

    dataset_abstraction = mlflow.data.from_pandas(
        transactions_post_covid_nouvelle_acquitaine.to_pandas()
    )
    mlflow.log_input(dataset_abstraction)

2024/10/28 17:53:46 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
c:\Users\Zakaria\Desktop\Data Science\Openclassrooms\supervisedlearning\.venv\Lib\site-packages\mlflow\types\utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [85]:
run_metrics = mlflow.search_runs(
    filter_string="""
    tags.mlflow.runName = 'pre_covid_catboost_Nouvelle-Aquitaine_post_covid_data' 
    AND status = 'FINISHED'
    """
)
run_metrics = run_metrics[[col for col in run_metrics.columns if col.startswith("metrics.")]]

In [86]:
run_metrics

,metrics.f1,metrics.precision,metrics.recall
0,0.515822,0.650209,0.427471


We then create our Run in the same way as in the course, using the new feature set and data.

In [89]:
features_2

In [91]:

with mlflow.start_run(run_name="catboost_Nouvelle-Aquitaine_post_covid") as run:

    X = transactions_post_covid_nouvelle_acquitaine.drop(
        [REGRESSION_TARGET, CLASSIFICATION_TARGET]
    ).to_pandas()
    y_classification = transactions_post_covid_nouvelle_acquitaine[CLASSIFICATION_TARGET].to_pandas()

    catboost_model = CatBoostClassifier(random_state=random_state, verbose=False)
    classification_scoring_metrics = ["recall", "precision", "f1"]

    scores, scores_dict, catboost_model = perform_cross_validation(
        X=X[features_2],
        y=y_classification,
        model=catboost_model,
        cross_val_type=StratifiedKFold(),
        scoring_metrics=classification_scoring_metrics,
    )

    mlflow.log_param("random_state", random_state)
    mlflow.log_param("features", features_2)

    for metric, value in scores_dict.items():
        mlflow.log_metric(metric, value)

    mlflow.sklearn.log_model(catboost_model, "catboost_classifier_post_covid")

    dataset_abstraction = mlflow.data.from_pandas(
        transactions_post_covid_nouvelle_acquitaine.to_pandas()
    )
    mlflow.log_input(dataset_abstraction)

feature_importances = catboost_model.get_feature_importance(Pool(X[features_2]))
most_important_features = get_features_most_importance(
    feature_importances, features_2
)


2024/10/28 18:11:08 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
c:\Users\Zakaria\Desktop\Data Science\Openclassrooms\supervisedlearning\.venv\Lib\site-packages\mlflow\types\utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Instead of using the MLflow UI, we will programmatically search for the best model among the post-Covid and pre-Covid runs. 

In [93]:
run_metrics_new_model = mlflow.search_runs(
    filter_string="""
    tags.mlflow.runName = 'catboost_Nouvelle-Aquitaine_post_covid'
    AND status = 'FINISHED'
    """
)
run_metrics_new_model = run_metrics_new_model[[col for col in run_metrics_new_model.columns if col.startswith("metrics.")]]

In [94]:
run_metrics_new_model

,metrics.train_precision_std,metrics.average_train_recall,metrics.average_test_precision,metrics.average_test_recall,metrics.train_recall_std,metrics.average_train_precision,metrics.test_precision_std,metrics.test_f1_std,metrics.average_train_f1,metrics.average_test_f1,metrics.train_f1_std,metrics.test_recall_std
0,0.006768,0.552081,0.436451,0.465252,0.01664,0.819325,0.101887,0.10322,0.659542,0.442434,0.012896,0.138184
